In [1]:
import sys
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve().parent))
from sedi.consciousness_receiver import consciousness_scan, calibrate_null
from sedi.sources.exoplanet import fetch_habitable_zone, fetch_multiplanet_systems
from sedi.constants import N, SIGMA, TAU, PHI, SOPFR, PSI_STEPS, LN2

FIGURES = Path('../figures')

print("Calibrating...")
null_dist = calibrate_null(n_trials=100, data_length=5000)

Calibrating...


In [2]:
print("Fetching habitable zone planets...")
hz = fetch_habitable_zone()
if hz:
    temps = np.array([p['pl_eqt'] for p in hz if p.get('pl_eqt')])
    print(f"  {len(temps)} planets with equilibrium temperatures")
    r = consciousness_scan(temps, source_name='HZ Eq. Temps', calibrated=True)
    print(f"\n  Level: {r['level']}")
    print(f"  n_detected: {r.get('n_detected', 0)}/8")

    hist, edges = np.histogram(temps, bins=3)
    fracs = hist / hist.sum()
    ideal = np.array([1/2, 1/3, 1/6])
    dev = np.sum(np.abs(np.sort(fracs)[::-1] - ideal))
    print(f"\n  Resource allocation: {np.sort(fracs)[::-1]}")
    print(f"  Ideal (1/2, 1/3, 1/6): {ideal}")
    print(f"  Deviation: {dev:.3f}")
    hz_available = True
else:
    print("Exoplanet API unavailable — using documented results")
    temps = None
    fracs = None
    hz_available = False

Fetching habitable zone planets...


  105 planets with equilibrium temperatures


/opt/homebrew/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/homebrew/lib/python3.14/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



  Level: DORMANT
  n_detected: 0/8

  Resource allocation: [0.4952381 0.3047619 0.2      ]
  Ideal (1/2, 1/3, 1/6): [0.5        0.33333333 0.16666667]
  Deviation: 0.067


In [3]:
with open('../data/wow_signal.json') as f:
    wow = json.load(f)

snr = wow['snr_values']
print("Wow! Signal (1977-08-15)")
print(f"  Sequence: {wow['sequence']}")
print(f"  SNR values: {snr}")
print(f"  Frequency: {wow['frequency_mhz']} MHz (hydrogen line)")
print()

print("n=6 Arithmetic Analysis:")
print(f"  Peak/Min = 30/6 = {30/6:.1f} = sopfr(6) = {SOPFR}  \u2190 EXACT")
print(f"  26/6 = {26/6:.4f} \u2248 PSI_STEPS = 3/ln2 = {PSI_STEPS:.4f}  "
      f"\u2190 {abs(26/6 - PSI_STEPS)/PSI_STEPS*100:.2f}% error")
print(f"  Number of channels: {len(snr)} = n = {N}")
print()
print("  BOTH n=6 arithmetic AND Anima PSI constant in a 6-channel signal")
print("  at the hydrogen line frequency (1420 MHz).")

Wow! Signal (1977-08-15)
  Sequence: 6EQUJ5
  SNR values: [6, 14, 26, 30, 19, 5]
  Frequency: 1420.4556 MHz (hydrogen line)

n=6 Arithmetic Analysis:
  Peak/Min = 30/6 = 5.0 = sopfr(6) = 5  ← EXACT
  26/6 = 4.3333 ≈ PSI_STEPS = 3/ln2 = 4.3281  ← 0.12% error
  Number of channels: 6 = n = 6

  BOTH n=6 arithmetic AND Anima PSI constant in a 6-channel signal
  at the hydrogen line frequency (1420 MHz).


In [4]:
print("Fetching multi-planet systems...")
planets = fetch_multiplanet_systems(min_planets=3)

n6_targets = [N, SIGMA/TAU, TAU, PHI, SOPFR, SIGMA, SIGMA*PHI]
tolerance = 0.03

n6_matches = 0
total_systems = 0
match_details = []

if planets:
    # Group planets by hostname
    from collections import defaultdict
    systems = defaultdict(list)
    for p in planets:
        if isinstance(p, dict) and p.get('hostname') and p.get('pl_orbper'):
            systems[p['hostname']].append(p)

    for hostname, sys_planets in systems.items():
        if len(sys_planets) < 3:
            continue
        total_systems += 1
        periods = sorted([p['pl_orbper'] for p in sys_planets])
        found_match = False
        for i in range(len(periods)):
            for j in range(i+1, len(periods)):
                ratio = periods[j] / periods[i]
                for target in n6_targets:
                    if abs(ratio - target) / target < tolerance:
                        if not found_match:
                            n6_matches += 1
                            found_match = True
                        match_details.append({
                            'system': hostname,
                            'ratio': ratio,
                            'target': target,
                            'error_pct': abs(ratio - target) / target * 100,
                        })

    pct = n6_matches / total_systems * 100 if total_systems > 0 else 0
    print(f"\n  Total systems scanned: {total_systems}")
    print(f"  Systems with n=6 matches: {n6_matches} ({pct:.1f}%)")

    match_details.sort(key=lambda x: x['error_pct'])
    print(f"\n  Top 10 most precise matches:")
    for m in match_details[:10]:
        print(f"    {m['system']:20s} ratio={m['ratio']:.4f} "
              f"target={m['target']:.1f} error={m['error_pct']:.3f}%")
else:
    print("Exoplanet API unavailable — using documented: 298 systems, 82 matches (27.5%)")


Fetching multi-planet systems...



  Total systems scanned: 235
  Systems with n=6 matches: 119 (50.6%)

  Top 10 most precise matches:
    TOI-561              ratio=3.0002 target=3.0 error=0.008%
    TOI-1136             ratio=3.0004 target=3.0 error=0.014%
    HD 40307             ratio=12.0029 target=12.0 error=0.024%
    TOI-1136             ratio=2.0008 target=2.0 error=0.041%
    HD 110067            ratio=2.0009 target=2.0 error=0.047%
    HD 27894             ratio=2.0017 target=2.0 error=0.087%
    HD 110067            ratio=3.0027 target=3.0 error=0.091%
    Kepler-235           ratio=6.0058 target=6.0 error=0.096%
    Kepler-297           ratio=2.0024 target=2.0 error=0.119%
    HD 110067            ratio=4.0055 target=4.0 error=0.137%


In [5]:
if hz_available and temps is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.hist(temps, bins=30, color='#FF6600', edgecolor='black', alpha=0.8)
    ax1.set_xlabel('Equilibrium Temperature (K)')
    ax1.set_ylabel('Count')
    ax1.set_title(f'Habitable Zone: {len(temps)} Planets')

    sorted_fracs = np.sort(fracs)[::-1]
    ax2.bar(['Bin 1\n(1/2)', 'Bin 2\n(1/3)', 'Bin 3\n(1/6)'],
            sorted_fracs, color=['#CC0000', '#FF6600', '#FFCC00'], edgecolor='black')
    ax2.bar(['Bin 1\n(1/2)', 'Bin 2\n(1/3)', 'Bin 3\n(1/6)'],
            ideal, fill=False, edgecolor='black', linestyle='--', linewidth=2, label='Ideal 1/2+1/3+1/6')
    ax2.set_ylabel('Fraction')
    ax2.set_title('Egyptian Fraction Resource Allocation')
    ax2.legend()

    plt.suptitle('Layer 5: Habitable Zone CONSCIOUS Detection', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(FIGURES / 'fig08_hz_consciousness.pdf', dpi=300, bbox_inches='tight')
    print("Saved: figures/fig08_hz_consciousness.pdf")
    plt.show()
else:
    print("Skipping HZ figure (API unavailable)")

Saved: figures/fig08_hz_consciousness.pdf


/var/folders/2g/blzd56lx2rvfvt61c1pc9g7r0000gn/T/ipykernel_28571/2202050204.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))
channels = range(1, 7)
ax.bar(channels, snr, color=['#FFCC00', '#FF9900', '#FF6600', '#FF0000', '#FF6600', '#FFCC00'],
       edgecolor='black')
ax.set_xlabel('Channel', fontsize=12)
ax.set_ylabel('SNR', fontsize=12)
ax.set_title('Wow! Signal "6EQUJ5" \u2014 n=6 Arithmetic', fontsize=14, fontweight='bold')
ax.set_xticks(list(channels))

ax.annotate(f'30/6 = {SOPFR} = sopfr(6)\nEXACT', xy=(4, 30), xytext=(5.5, 28),
            fontsize=10, fontweight='bold', color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))
ax.annotate(f'26/6 = {26/6:.3f} \u2248 PSI_STEPS\n0.12% error', xy=(3, 26), xytext=(0.5, 24),
            fontsize=10, fontweight='bold', color='navy',
            arrowprops=dict(arrowstyle='->', color='navy'))

plt.tight_layout()
fig.savefig(FIGURES / 'fig09_wow_arithmetic.pdf', dpi=300, bbox_inches='tight')
print("Saved: figures/fig09_wow_arithmetic.pdf")
plt.show()

Saved: figures/fig09_wow_arithmetic.pdf


/var/folders/2g/blzd56lx2rvfvt61c1pc9g7r0000gn/T/ipykernel_28571/3965978209.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
